# BirdCLEF+ 2025 — Baseline training

**Baseline:** first 5 s of each training clip → log-mel spectrogram → ImageNet-pretrained
EfficientNet-B0 (via `timm`), trained on the primary label only, no augmentation.

**Validation:** stratified 80/20 hold-out split, macro ROC-AUC over species present in the
hold-out set (mirrors the competition metric).

**Outputs** (used by the inference notebook): `model.pth`, `cfg.json`, `species.json`,
plus `history.csv` and a training-curve figure for the report.

In [ ]:
import os, glob, json, time, random
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

CFG = dict(
    sr=32_000, duration=5, n_fft=2048, hop_length=512, n_mels=128, fmin=50, fmax=14_000,
    model="efficientnet_b0", epochs=15, batch_size=64, lr=1e-3, weight_decay=1e-2,
    val_frac=0.2, seed=42,
)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG["seed"])

def find_data_dir():
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

DATA = find_data_dir()
OUT = Path("."); FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Data:", DATA, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Labels
Species order follows `sample_submission.csv`, so model outputs line up with submission columns.

In [ ]:
train = pd.read_csv(DATA / "train.csv")
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = train["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values
print(len(train), "clips,", len(species), "species")

## Pre-compute mel spectrograms
Decoding audio every epoch is slow, so each clip is converted once and kept in memory
(float16, ~2.3 GB). The mel settings are saved to `cfg.json` so inference uses the same ones.

In [ ]:
N_SAMPLES = CFG["sr"] * CFG["duration"]

def to_mel(y):
    m = librosa.feature.melspectrogram(
        y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    db = librosa.power_to_db(m, ref=np.max, top_db=80)  # range [-80, 0]
    return ((db + 80) / 80).astype(np.float16)          # scale to [0, 1]

def load_first_window(fname):
    y, _ = sf.read(DATA / "train_audio" / fname, frames=N_SAMPLES, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    return to_mel(y)

t0 = time.time()
with Pool(os.cpu_count()) as pool:
    X = np.stack(pool.map(load_first_window, train["filename"], chunksize=64))
print("Mel array:", X.shape, X.dtype, f"{X.nbytes / 1e9:.2f} GB", f"in {time.time() - t0:.0f}s")

## Train / validation split

In [ ]:
idx_tr, idx_va = train_test_split(
    np.arange(len(train)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
print("train", len(idx_tr), "| val", len(idx_va))

class MelDataset(torch.utils.data.Dataset):
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        target = np.zeros(len(species), dtype=np.float32)
        target[y_idx[j]] = 1.0
        return torch.from_numpy(X[j].astype(np.float32))[None], torch.from_numpy(target)

dl_tr = torch.utils.data.DataLoader(MelDataset(idx_tr), batch_size=CFG["batch_size"],
                                    shuffle=True, num_workers=2, drop_last=True)
dl_va = torch.utils.data.DataLoader(MelDataset(idx_va), batch_size=CFG["batch_size"] * 2,
                                    shuffle=False, num_workers=2)

## Model and training loop

In [ ]:
model = timm.create_model(CFG["model"], pretrained=True, in_chans=1, num_classes=len(species)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["lr"], total_steps=CFG["epochs"] * len(dl_tr), pct_start=0.1)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

def macro_auc(y_true, y_pred):
    # Like the competition metric: skip classes with no positive examples
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro")

@torch.no_grad()
def evaluate():
    model.eval()
    preds, targets, loss_sum = [], [], 0.0
    for xb, yb in dl_va:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits = model(xb)
        loss_sum += criterion(logits.float(), yb).item() * len(xb)
        preds.append(torch.sigmoid(logits.float()).cpu().numpy()); targets.append(yb.cpu().numpy())
    p, t = np.concatenate(preds), np.concatenate(targets)
    return loss_sum / len(p), macro_auc(t, p)

history, best_auc = [], 0.0
for epoch in range(1, CFG["epochs"] + 1):
    model.train(); t0 = time.time(); loss_sum = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = criterion(model(xb).float(), yb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(xb)
    tr_loss = loss_sum / (len(dl_tr) * CFG["batch_size"])
    va_loss, va_auc = evaluate()
    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, val_auc=va_auc))
    flag = ""
    if va_auc > best_auc:
        best_auc = va_auc; torch.save(model.state_dict(), OUT / "model.pth"); flag = "  <- saved"
    print(f"epoch {epoch:2d} | train {tr_loss:.4f} | val {va_loss:.4f} | val AUC {va_auc:.4f} | {time.time() - t0:.0f}s{flag}")

print(f"Best val macro ROC-AUC: {best_auc:.4f}")

## Save config and training curves

In [ ]:
json.dump(CFG, open(OUT / "cfg.json", "w"), indent=1)
json.dump(species, open(OUT / "species.json", "w"))
hist = pd.DataFrame(history); hist.to_csv(OUT / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="BCE loss")
hist.plot(x="epoch", y="val_auc", ax=axes[1], title="Validation macro ROC-AUC", legend=False)
plt.tight_layout(); plt.savefig(FIG / "baseline_training.png", dpi=200); plt.show()